In [8]:
import string
all_letters = string.ascii_letters + " .,;'"
n_letters = len(all_letters)

import unicodedata
# Turn a Unicode string to plain ASCII
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

# Build the category_lines dictionary, a list of names per language
category_lines = {}
all_categories = []

import os
import glob
from io import open
for filename in glob.glob('../data-sets/data/names/*.txt'):
    category = os.path.splitext(os.path.basename(filename))[0]
    lines = open(filename, encoding='utf-8').read().strip().split('\n')
    lines = [unicodeToAscii(line) for line in lines]
    category_lines[category] = lines
    all_categories.append(category)

n_categories = len(category_lines.keys())

print('Amount of categories:', n_categories)
print('Some Dutch examples:', category_lines['Dutch'][:5])

Amount of categories: 18
Some Dutch examples: ['Aalsburg', 'Aalst', 'Aarle', 'Achteren', 'Achthoven']


In [9]:
char_to_num = {char: num for num, char in enumerate(all_letters)}
num_to_char = {num: char for char, num in char_to_num.items()}

cat_to_num = {cat: num for num, cat in enumerate(all_categories)}
num_to_cat = {num: cat for cat, num in cat_to_num.items()}

In [40]:
import torch
import random
import torch.nn.functional as F

pad_to_length = 25

def build_dataset(lines):
    X, Y = [], []

    for line in lines:
        enc = torch.tensor([char_to_num[c] for c in line])
        one_hot_enc = F.one_hot(enc, num_classes=n_letters)
        
        # Pad with zeros
        padding_length = max(0, pad_to_length - one_hot_enc.shape[0])
        pad_enc = F.pad(one_hot_enc, (0, 0, 0, padding_length), mode='constant', value=0)

        X.append(pad_enc)
        Y.append(torch.tensor([cat_to_num[key]]))
    
    X = torch.stack(X)
    Y = torch.stack(Y)

    return X, Y

Xtr, Ytr = [], []
Xte, Yte = [], []

for key, lines in category_lines.items():
    lines = random.sample(lines, len(lines))
    split = int(0.8 * len(lines))

    Xtr_part, Ytr_part = build_dataset(lines[:split])
    Xte_part, Yte_part = build_dataset(lines[split:])

    Xtr.append(Xtr_part)
    Ytr.append(Ytr_part)
    Xte.append(Xte_part)
    Yte.append(Yte_part)

Xtr = torch.cat(Xtr, dim=0)
Ytr = torch.cat(Ytr, dim=0)
Xte = torch.cat(Xte, dim=0)
Yte = torch.cat(Yte, dim=0)

ix = random.randint(0, Xtr.shape[0] - 1)
print('Random example:', Xtr[ix].shape, '=>', Ytr[ix].item())
print('Amount of train examples:', Xtr.shape[0])

Random example: torch.Size([25, 57]) => 16
Amount of train examples: 16053


In [33]:
import torch.nn as nn

n_hidden = 128

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(RNN, self).__init__()

        self.hidden_size = hidden_size

        self.i2h = nn.Linear(input_size + hidden_size, hidden_size)
        self.h2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, input, hidden):
        combined = torch.cat((input, hidden), 1)
        hidden = self.i2h(combined)
        output = self.h2o(hidden)
        output = self.softmax(output)
        return output, hidden

    def initHidden(self):
        return torch.zeros(1, self.hidden_size)

model = RNN(Vs, n_hidden, n_categories)

In [38]:
input = lineToTensor('Albert')
hidden = torch.zeros(1, n_hidden)

output, next_hidden = model(input[0], hidden)
print(output)

def categoryFromOutput(output):
    top_n, top_i = output.topk(1)
    category_i = top_i[0].item()
    return list(category_lines.keys())[category_i], category_i

print(categoryFromOutput(output))

tensor([[-2.8850, -2.8471, -2.8237, -2.9331, -2.9011, -2.8694, -2.9706, -2.8577,
         -2.9461, -2.8836, -3.0241, -2.8456, -2.8527, -2.7967, -2.8736, -2.9809,
         -2.9859, -2.7881]], grad_fn=<LogSoftmaxBackward0>)
('Korean', 17)
